# 原始 Stage1 0bp 恢复必要条件与首小时带方向成交量来源交集

只核对原始 A_0 自有状态 10:32 恢复动作的固定必要条件和来源覆盖；不读取净动作标签、不改策略或阈值。

In [1]:
import os,json,hashlib,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
from src.data.nas_market_data import archive_nas_recorder,_settings
root=Path.cwd();card=root/'research/stage1_0bp_signed_volume_action_route_preflight_20261023';plan=json.loads((card/'plan.json').read_text())
def sha(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and not (card/'run_state.json').exists() and not (card/'qlib_recorder').exists()
ps=plan['parent'];fs=plan['feature_source'];assert sha(root/ps['summary'])==ps['summary_sha256'] and sha(root/fs['summary'])==fs['summary_sha256']
psum=json.loads((root/ps['summary']).read_text());fsum=json.loads((root/fs['summary']).read_text());assert psum['recorder_id']==ps['recorder_id'] and fsum['recorder_id']==fs['recorder_id']
assert psum['nas_evidence']['sha256']==ps['NAS_archive_sha256'] and fsum['nas_evidence']['sha256']==fs['NAS_archive_sha256']
prb=json.loads((root/Path(ps['summary']).parent/'readback_verification.json').read_text());frb=json.loads((root/Path(fs['summary']).parent/'nas_scoped_restore.json').read_text());assert prb['status']=='passed' and prb['archive_sha256']==ps['NAS_archive_sha256'] and prb['all_artifact_hashes_match'];assert frb['status']=='verified' and frb['archive_sha256']==fs['NAS_archive_sha256'] and frb['all_sha256_and_sizes_match']
pa=Path(ps['recorder_uri'])/ps['experiment_id']/ps['recorder_id']/'artifacts';fa=Path(fs['recorder_uri'])/fs['experiment_id']/fs['recorder_id']/'artifacts'
for name,digest in ps['artifact_sha256'].items():assert sha(pa/name)==digest
assert sha(fa/fs['artifact'])==fs['artifact_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/plan['calendar_provider']),region='us',kernels=1)
calendar=pd.DatetimeIndex(D.calendar(start_time=plan['matched_trade_dates'][0],end_time=plan['matched_trade_dates'][1],freq='day'));assert len(calendar)==plan['expected_dates']
with R.uri_context(ps['recorder_uri']):
 r=R.get_recorder(recorder_id=ps['recorder_id'],experiment_id=ps['experiment_id']);native=r.load_object('native/A_active_research_replay_0');frame=r.load_object('inputs/frame');selection=r.load_object('inputs/selection');recovery=r.load_object('inputs/recovery_spec')
with R.uri_context(fs['recorder_uri']):feature_source=R.get_recorder(recorder_id=fs['recorder_id'],experiment_id=fs['experiment_id']).load_object(fs['artifact'])
a=native['authority'].copy();assert len(a)==len(frame)==len(calendar)==944 and not a.recovery_action.any()
for x in (a,frame,feature_source):x['trade_date']=pd.to_datetime(x.trade_date).dt.normalize()
assert a.trade_date.equals(frame.trade_date) and a.trade_date.equals(pd.Series(calendar,name='trade_date'))
joined=a[['trade_date','portfolio_drawdown','current_beta_cap','risk_state','risk_clock','recovery_action']].merge(frame[['trade_date','soxl_weight','soxx_weight','alpha_probability']],on='trade_date',validate='one_to_one').merge(feature_source[['trade_date','pressure','status','assumed_available_at','receiver_decision_at']],on='trade_date',validate='one_to_one')
assert len(joined)==944 and joined.trade_date.equals(pd.Series(calendar,name='trade_date'))
raw_beta=3*joined.soxl_weight.astype(float)+joined.soxx_weight.astype(float);target_beta=np.minimum(raw_beta,float(selection['action_beta']))
joined['raw_beta']=raw_beta;joined['fixed_action_beta_cap']=float(selection['action_beta']);joined['target_beta']=target_beta
joined['drawdown_gate']=joined.portfolio_drawdown.ge(float(recovery['recovery_eligibility_drawdown']));joined['target_above_cap_gate']=target_beta.gt(joined.current_beta_cap+1e-12);joined['alpha_floor_gate']=joined.alpha_probability.ge(float(selection['minimum_source_alpha_probability']))
joined['necessary_route']=joined.drawdown_gate&joined.target_above_cap_gate&joined.alpha_floor_gate
joined['source_known']=joined.status.eq('known')&joined.pressure.notna()&np.isfinite(joined.pressure.astype(float));joined['necessary_route_source_known']=joined.necessary_route&joined.source_known
assert joined.source_known.all();assert (pd.to_datetime(joined.assumed_available_at)<pd.to_datetime(joined.receiver_decision_at)).all()
joined['quarter']=joined.trade_date.dt.to_period('Q').astype(str);joined['year']=joined.trade_date.dt.year
static=pd.concat({'feature':joined.set_index('trade_date')[['pressure']].rename(columns={'pressure':'SOXX_first_hour_signed_volume_pressure'}),'meta':joined.set_index('trade_date')[['drawdown_gate','target_above_cap_gate','alpha_floor_gate','necessary_route','source_known']]},axis=1)
static.index=pd.MultiIndex.from_arrays([joined.trade_date,['SOXL_SOXX_CASH']*len(joined)],names=['datetime','instrument'])
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'held':tuple(plan['matched_trade_dates'])});prepared=dataset.prepare('held',col_set=['feature','meta']);assert len(prepared)==944
quarters=[{'quarter':k,'sessions':len(g),'drawdown_gate':int(g.drawdown_gate.sum()),'target_above_cap_gate':int(g.target_above_cap_gate.sum()),'alpha_floor_gate':int(g.alpha_floor_gate.sum()),'necessary_route':int(g.necessary_route.sum()),'necessary_route_source_known':int(g.necessary_route_source_known.sum())} for k,g in joined.groupby('quarter',sort=True)]
years=[{'year':int(k),'sessions':len(g),'complete_year':k in (2024,2025),'necessary_route':int(g.necessary_route.sum()),'source_known':int(g.source_known.sum())} for k,g in joined.groupby('year',sort=True)]
summary={'experiment':card.name,'plan_sha256':sha(card/'plan.json'),'decision':'read_only_original_Stage1_0bp_recovery_necessary_route_source_overlap_no_label','objective_achieved':False,'source_parent_recorder_id':ps['recorder_id'],'source_feature_recorder_id':fs['recorder_id'],'matched_sessions':len(joined),'component_counts':{k:int(joined[k].sum()) for k in ('drawdown_gate','target_above_cap_gate','alpha_floor_gate','necessary_route','source_known','necessary_route_source_known')},'actual_recovery_actions':int(joined.recovery_action.sum()),'quarterly':quarters,'annual':years,'source_first_receipt_and_finality_qualified':False,'necessary_not_sufficient':True,'new_model_fits':0,'new_native_accounts':0,'outcome_label_reads':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/daily']}}
state={'status':'started','plan_sha256':summary['plan_sha256'],'parent_loads':4,'feature_loads':1,'new_model_fits':0,'new_native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
tracking=card/'qlib_recorder'
with R.start(experiment_name=card.name,uri=str(tracking)):
 rec=R.get_recorder();state.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
 dataset.config(dump_all=True,recursive=True);rec.save_objects(artifact_path='inputs',plan=plan,dataset=dataset,source_summary_hashes={'parent':ps['summary_sha256'],'feature':fs['summary_sha256']});rec.save_objects(artifact_path='analysis',daily=joined)
 rec.log_metrics(necessary_route=int(joined.necessary_route.sum()),necessary_route_source_known=int(joined.necessary_route_source_known.sum()),new_model_fits=0,new_native_accounts=0)
 with R.uri_context(str(tracking)):reread=R.get_recorder(recorder_id=rec.id,experiment_id=str(rec.experiment_id)).load_object('analysis/daily')
 assert reread.equals(joined)
summary.update(recorder_id=state['recorder_id'],experiment_id=state['experiment_id'],recorder_uri=str(tracking));(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':card.name,'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=state['recorder_id'],experiment_id=state['experiment_id'])
with tempfile.TemporaryDirectory(prefix='stage1-0bp-route-restore-') as temp:
 archive=Path(temp)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as source:
  names=[x.name for x in source.getmembers() if x.isfile()];assert any(x.endswith('/artifacts/analysis/daily') for x in names) and any(x.endswith('/artifacts/inputs/dataset') for x in names)
summary['nas_evidence']=evidence;summary['nas_readback']={'sha256':evidence['sha256'],'verified':True};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');state['status']='completed';state['NAS_archive_sha256']=evidence['sha256'];(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
print({'decision':summary['decision'],'component_counts':summary['component_counts'],'recorder_id':summary['recorder_id'],'NAS_sha256':evidence['sha256']})


2026/10/23 04:20:20 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[80156:MainThread](2026-10-23 04:20:21,184) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[80156:MainThread](2026-10-23 04:20:21,185) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[80156:MainThread](2026-10-23 04:20:21,185) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/stage1_domain_heads_20261018')}


...

[80156:MainThread](2026-10-23 04:20:22,003) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name stage1_0bp_signed_volume_action_route_preflight_20261023.


[80156:MainThread](2026-10-23 04:20:22,005) INFO - qlib.workflow - [exp.py:258] - Experiment 281538982284140456 starts running ...


[80156:MainThread](2026-10-23 04:20:22,085) INFO - qlib.workflow - [recorder.py:345] - Recorder e57390b08fce48b8bdccaa275438e4f8 starts running under Experiment 281538982284140456 ...


[80156:MainThread](2026-10-23 04:20:22,885) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{'decision': 'read_only_original_Stage1_0bp_recovery_necessary_route_source_overlap_no_label', 'component_counts': {'drawdown_gate': 494, 'target_above_cap_gate': 199, 'alpha_floor_gate': 944, 'necessary_route': 199, 'source_known': 944, 'necessary_route_source_known': 199}, 'recorder_id': 'e57390b08fce48b8bdccaa275438e4f8', 'NAS_sha256': '6c04907ff89341890cb25fe113d9b8b376457c1298b9ec8978460fd4a8bd92d2'}


# 原始 Stage1 0bp 恢复必要条件与首小时带方向成交量来源交集

结论：原始 Stage1 `A_active_research_replay_0` 的 944 个复用历史动作日中，固定回撤、目标高于风险上限和来源概率下限三项必要条件同时成立 199 日；新首小时带方向成交量坐标在全部 199 日已知。旧恢复动作实际触发 0 次。本卡没有拟合、标签或新账户。

分解：回撤条件成立 494 日，目标高于风险上限 199 日，来源概率下限 944 日。这里只复用原始 A_0 自有状态，不借用 beta-composition 候选的风险路径。

区间对比：2024Q3 有 10 个必要动作日，2025Q2 有 33 日，2026Q3 有 53 日。原 Stage1 在 2024Q3 的现金保护有效、2026Q3 的现金机会损失较重，但必要条件数量不能证明交易方向或净收益；这些季度不用于设阈值。

来源的 10:31 可得时间是历史 bar 完结假设，实际首次接收和最终版本时钟尚未认证。199 日未核对旧恢复模型的全部 21 项特征、模型概率及真实 0bp 净动作优势，因此只是可决策样本的上界；旧 0.825 门槛保持冻结。训练来源仅有 2018–2022 的 1,060 个已知日，2011–2018 配置头的分钟缺口仍在。

原生 Qlib Recorder `e57390b08fce48b8bdccaa275438e4f8` 与 NAS 回读 SHA-256 `6c04907ff89341890cb25fe113d9b8b376457c1298b9ec8978460fd4a8bd92d2` 已验证。下一步应在原始 Stage1 的 0bp 自有路径上重建同状态、同次日目标复位、仍扣 IBKR Tiered 的 beta0.6 恢复/HOLD 净优势标签，并核对训练标签成熟和全部旧恢复输入；只有其固定信息检验通过，才运行完整 0bp Stage1 派生账户。